In [ ]:
import sys
sys.path.insert(0, "..") # dalgebra is here
from dalgebra import *
%display latex

def LB(A,B):
    return A*B - B*A

## TODO: to kp

# Stationary **KP Hierarchies**

In the paper by [Aratyn et. al](https://arxiv.org/abs/hep-th/9209006), they define the *KP hierarchy* as the conditions in some differential variables $u_i$ such that the operator
$$L = \partial + u_1 \partial^{-1} + u_2\partial^{-2} + ...,$$
satisfies the commutation rule:
$$[L, (L^r)_+] = 0.$$

**Some guesses**

* For any $r$, $[L, (L^r)_+]$ has order $-1$, generically.
* For any $r$, the coefficient $[\partial^{-i}][L, (L^r)_+] = ru_{i+r} + B(u_1,\ldots,u_{i+r-1})$.
* For a fixed $r$, we can solve for $u_r$ on by doing one integration. We can do this when substituing previous solutions to further equations only.
* Once a fixed $r_0$ has been solved, we can enforce further commutations with $r>r_0$. If so, only the first equation(s) are necessary for having the commutation (the following are implied by the first conditions).

This means that if we fix a value $n > 0$ to be the first order of commutation, this provides an expression of $L$ in a finite amount of differential variables (similar to the GD hierarchies we start with an $L$ with fixed amount of differential variables. Then the hierarchy for order $n$ will be the sequence of conditions for $m > n$ to have a commutation (this will be the level $m$ for the $n$th hierarchy).

In [14]:
def Pow_TR(L, p):
    return (L^p).differential_part()

def solve_at_order(L, r, bound):
    u = [0] + [L.parent().base().gen(f"u_{i}") for i in range(1,bound+1)]
    R = L.parent(); D = R.gen(); Di = R.igen()
    sols = dict()
    lb = LB(L, Pow_TR(L,r))
    
    for i in range(1,bound-r+2):
        sols[f"u_{r+i-1}"] = lb[-i](**sols).solve(u[r+i-1])
    return D + sum(sols.get(f"u_{i}",u[i])*Di**i for i in range(1,len(u))), sols

def equations_LB(L, r, bound):
    lb = LB(L, Pow_TR(L, r))
    return {i : lb[i] for i in range(-1, -bound+r-2, -1) if lb[i] != 0}

def check_equations(equations, n, rank):
    rem_zero = dict()
    full_rems = dict()
    for m in equations.keys():
        if len(equations[m]) == 0:
            rem_zero[m] = True
        elif len(equations[m]) < n-1:
            rem_zero[m] = False
        else:
            sort = sorted(equations[m],reverse=True) # keys are sorted
            A = rank.autoreduced(*[equations[m][i] for i in sort[:n]]) # getting the first (n-1) equations
            full_rems[m] = {i : rank.remainder(equations[m][i], A)[0] for i in equations[m]}
            rem_zero[m] = all(rem == 0 for rem in full_rems[m].values())
    return all(rem_zero.values()), full_rems

In [7]:
O = 20
R = DifferentialPolynomialRing(QQ, names=[f"u_{i}" for i in range(1,O+1)])
u = ([0],) + R.gens()
S = PseudoDOperator_Ring(R, "D")
D = S.gen(); Di = S.igen()
L = D + sum(u[i][0]*Di**i for i in range(len(u)))
rank = R.ranking()

### Case for $n=2$

Let us check all the guesses for the case of $n=2$. Since $L$ has an infinite tail, we need to fix an order for it (will be `O`) and then we can do computations up to this element.

#### We use the solver above
The fact that this method works is because the guess 3 is working

In [8]:
L_eval, sols = solve_at_order(L, 2, O)

#### We check the fact that only 1 equation is needed at each level
We now fix a new level $m > 2$ and we compute the lie bracket of the evaluated operator `L_eval` with the corresponding power of `L_eval`. Then we will collect the coefficients and check that we can reduce all coefficients with the first.

This can be done using a ranking function.

In [10]:
%%time
equations = {m: equations_LB(L_eval, m, O) for m in range(3,8)}

In [15]:
%%time
all_zero, rems = check_equations(equations, 2, rank)
all_zero

CPU times: user 12.3 s, sys: 189 ms, total: 12.4 s
Wall time: 12.4 s


True

### Case for $n=3$

This will be the equivalent to the Bousinesq hierarchy. Hence, I would expect that we will end up with 2 variables ($u_1$ and $u_2$), and two equations for each level will be necessary to get a full cancellation.

#### We use the solver above
The fact that this method works is because the guess 3 is working

In [16]:
L_eval, sols = solve_at_order(L, 3, O)

#### We check that 2 equations are needed in this case

In [17]:
%%time
equations = {m: equations_LB(L_eval, m, O) for m in range(4,9)}

CPU times: user 4min 45s, sys: 944 ms, total: 4min 46s
Wall time: 4min 46s


In [ ]:
%%time
all_zero, rems = check_equations(equations, 3, rank)
all_zero

# The second (stationary) **KP hierarchies**

Let us consider a generic operator $L$ in normal form of order $n$:
$$L = \partial^n + u_{n-2}\partial^{n-2} + \ldots + u_1\partial + u_0.$$

It is clear that is has a pseudo-differential operator of order $1$ that will be its $n$th root. This can always be computed in terms of $u_0,\ldots,u_{n-2}$ and its derivatives.

In [38]:
n = 2
O = 20
R = DifferentialPolynomialRing(QQ, names=[f"u_{i}" for i in range(n-1)]+[f"y_{j}" for j in range(O-1)])
u = R.gens()[:n-1]
y = R.gens()[n-1:]
S = PseudoDOperator_Ring(R, "D")
D = S.gen(); Di = S.igen()
L = D^n + sum(u[i][0]*D^i for i in range(len(u)))
rank = R.ranking()

In [39]:
L_p = L^(1/n)
L_p

D + (1/2*u_0_0)*D^(-1) + (-1/4*u_0_1)*D^(-2) + (-1/8*u_0_0^2 + 1/8*u_0_2)*D^(-3) + (3/8*u_0_0*u_0_1 - 1/16*u_0_3)*D^(-4) + (-7/16*u_0_0*u_0_2 + 1/16*u_0_0^3 - 11/32*u_0_1^2 + 1/32*u_0_4)*D^(-5) + (15/32*u_0_0*u_0_3 - 15/32*u_0_0^2*u_0_1 + 15/16*u_0_1*u_0_2 - 1/64*u_0_5)*D^(-6) + (85/64*u_0_0*u_0_1^2 - 31/64*u_0_0*u_0_4 + 55/64*u_0_0^2*u_0_2 - 5/128*u_0_0^4 - 37/32*u_0_1*u_0_3 - 91/128*u_0_2^2 + 1/128*u_0_6)*D^(-7) + (-175/32*u_0_0*u_0_1*u_0_2 + 63/128*u_0_0*u_0_5 - 175/128*u_0_0^2*u_0_3 + 35/64*u_0_0^3*u_0_1 + 175/128*u_0_1*u_0_4 - 175/128*u_0_1^3 + 245/128*u_0_2*u_0_3 - 1/256*u_0_7)*D^(-8) + (623/64*u_0_0*u_0_1*u_0_3 + 1589/256*u_0_0*u_0_2^2 - 127/256*u_0_0*u_0_6 - 805/256*u_0_0^2*u_0_1^2 + 511/256*u_0_0^2*u_0_4 - 175/128*u_0_0^3*u_0_2 + 7/256*u_0_0^5 - 405/256*u_0_1*u_0_5 + 2359/256*u_0_1^2*u_0_2 - 631/256*u_0_2*u_0_4 - 699/512*u_0_3^2 + 1/512*u_0_8)*D^(-9) + (-4053/256*u_0_0*u_0_1*u_0_4 + 2205/256*u_0_0*u_0_1^3 - 6195/256*u_0_0*u_0_2*u_0_3 + 255/512*u_0_0*u_0_7 + 2205/128*u_0_0^2*u_0_1*u_0_2 - 1407/512*u_0_0^2*u_0_5 + 735/256*u_0_0^3*u_0_3 - 315/512*u_0_0^4*u_0_1 - 11403/512*u_0_1*u_0_2^2 + 231/128*u_0_1*u_0_6 - 9219/512*u_0_1^2*u_0_3 + 399/128*u_0_2*u_0_5 + 945/256*u_0_3*u_0_4 - 1/1024*u_0_9)*D^(-10) + o(D^(-11))

Now we would like to check how much do $L$ and $L^{1/n}$ commute:

In [58]:
m = 5
P = D^m + sum(y[i][0]*D^i for i in range(m-1))
lb = LB(L_p, P)

In [60]:
sol = dict()
for i in range(m-2,-1,-1):
    equ = lb[i](**{k.variable_name(): v for (k,v) in sol.items()})
    sol[y[i]] = equ.solve(y[i])
P_eval = D^m + sum(sol[y[i]]*D^i for i in range(m-1))

In [61]:
lb_eval = LB(L_p, P_eval)
A = [lb_eval[-i] for i in range(1, n)]
[rank.remainder(lb_eval[j], *A)[0] for j in range(n,O)]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

# The relation between KP and GD

They both interpret the problem of having a non-trivial commutator in a different way. However, they are clearly related.

On one hand (as we saw earlier) the KP hierarchy starts from a generic pseudo operator of order 1 and force a commutation with its differential $n$th power. From this operator (which now only have $n-1$ differential variables), we check when it commutes also with its differential $m$th power.

On the other hand, the GD hierarchy starts from a generic differential operator of order $n$. This has, by default, $n-1$ differential variables. Then, we search the conditions for this operator to have a non-trivial commutator of order $m$. By the results by Wilson, this operator of order $m$ is $(L^{m/n})_+$.

If we compare the two approaches, we see a clear pattern that we can relate. Essentially:
* Let $M(v_1,\ldots,v_{n-1})$ be the order 1 pseudo-operator for the $n$th KP hierarchy. Then if it satisfies the KP hierarchy at level $m$ then the coefficients $(M^n)_+$ satisfies the GD hierarchy of order $n$ at level $m$.
* Let $L(u_0,\ldots,u_{n-2})$ be the generic $n$th order operator. If it satisfies the $GD$ hierarchy at level $m$ then the coefficients of $L^{1/n}$ satisfy the KP hierarchy of order $n$ at level $m$.

## Checking the equivalence

### KP $\implies$ GD

# K.P. hierarchies (OLD)

Let us work with the following definition:

The KP-hierarchy of order $n$ at level $m$ are the commutation conditions such that if we take an operator $L = \partial + \sum_{i > 0} u_i \partial^{-i}$, and we force that a differential operator of order $m$ ($Q_m$) commutes with $L^n$, i.e., the conditions on the $u_i$ such that:
$$ord([(L^n)_+,Q]) \leq n-2.$$

## 1. Building the set-up

In [5]:
import sys
sys.path.insert(0, "../") # dalgebra is here
from dalgebra import *

R.<u_1,u_2,u_3,y_1,y_2,y_3,y_4,y_5,y_6> = DifferentialPolynomialRing(QQ)
PDO = PseudoDOperatorRing(R, "D")
D, Di = PDO.D, PDO.Di

In [10]:
L = D + u_1[0]*Di + u_2[0]*Di^2 + u_3[0]*Di^3
L4 = (L**4).differential_part()

In [25]:
Q = D^5 + y_1[0]*D^3 + y_2[0]*D^2+y_3[0]*D + y_4[0]
LB = L4.lie_bracket(Q)
sols = dict()

In [26]:
eq1 = LB[6]
sols[y_1] = eq1.solve(y_1)

In [29]:
eq2 = LB[5](dic=sols)
sols[y_2] = eq2.solve(y_2)

In [32]:
eq3 = LB[4](dic=sols)
sols[y_3] = eq3.solve(y_3)

In [33]:
eq4 = LB[3](dic=sols)
sols[y_4] = eq4.solve(y_4)

In [34]:
sols

{y_1_*: 5*u_1_0,
 y_2_*: 10*u_1_1 + 5*u_2_0,
 y_3_*: 10*u_1_0^2 + 10*u_1_2 + 10*u_2_1 + 5*u_3_0,
 y_4_*: 25/2*u_1_0*u_1_1 + 5*u_1_0*u_2_0 + 15/4*u_1_3 + 5*u_2_2 + 5/2*u_3_1}

In [42]:
Q_eval = D^5 + sols[y_1]*D^3 + sols[y_2]*D^2+sols[y_3]*D + sols[y_4]
LB_eval = L4.lie_bracket(Q_eval)

In [43]:
[LB_eval[i] for i in range(3)]

[-195*u_1_0*u_1_1*u_1_2 - 50*u_1_0*u_1_1*u_2_1 - 60*u_1_0*u_1_1*u_3_0 + 10*u_1_0*u_1_2*u_2_0 - 9/2*u_1_0*u_1_5 + 20*u_1_0*u_2_0*u_2_1 - 5*u_1_0*u_2_4 - 10*u_1_0*u_3_3 - 50*u_1_0^2*u_1_3 - 40*u_1_0^2*u_2_2 - 40*u_1_0^2*u_3_1 - 120*u_1_0^3*u_1_1 - 15*u_1_1*u_1_4 + 20*u_1_1*u_2_0^2 - 10*u_1_1*u_2_3 - 25*u_1_1*u_3_2 + 20*u_1_1^2*u_2_0 - 45*u_1_1^3 - 35*u_1_2*u_1_3 - 30*u_1_2*u_2_2 - 40*u_1_2*u_3_1 - 20*u_1_3*u_2_1 - 20*u_1_3*u_3_0 - 1/4*u_1_7 - 10*u_2_0*u_2_3 - 10*u_2_0*u_3_2 - 60*u_2_1*u_2_2 - 40*u_2_1*u_3_1 - 30*u_2_2*u_3_0 - u_2_6 - 20*u_3_0*u_3_1 - 3/2*u_3_5,
 -120*u_1_0*u_1_1^2 - 10*u_1_0*u_1_4 - 10*u_1_0*u_2_3 - 20*u_1_0*u_3_2 - 60*u_1_0^2*u_1_2 - 40*u_1_1*u_1_3 - 40*u_1_1*u_2_2 - 50*u_1_1*u_3_1 - 40*u_1_2*u_2_1 - 30*u_1_2*u_3_0 - 30*u_1_2^2 - 10*u_1_3*u_2_0 - u_1_6 - 40*u_2_0*u_2_2 - 20*u_2_0*u_3_1 - 20*u_2_1*u_3_0 - 40*u_2_1^2 - 4*u_2_5 - 5*u_3_4,
 -15*u_1_0*u_1_3 - 20*u_1_0*u_2_2 - 20*u_1_0*u_3_1 - 60*u_1_0^2*u_1_1 - 35*u_1_1*u_1_2 - 30*u_1_1*u_2_1 - 20*u_1_1*u_3_0 - 10*u_1_2*u_2_